# DSA 210 — Part 3: Machine Learning Analysis
**Predicting Formula 1 Race Winners**  
Eldar Ismayilzada — 36222

This notebook applies machine learning to predict whether a driver will win a given race.

Three supervised classifiers are trained and compared:
- **Logistic Regression** 
- **Random Forest** 
- **XGBoost** 

Class imbalance (~4.2% winners) is handled by **class weighting** (built into sklearn/xgboost).  
All models are evaluated with **5-fold cross-validation** and standard classification metrics (Recitation 8).

## 1. Imports & Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load & Inspect Data

In [3]:
# Update path if running locally
df = pd.read_csv(r'data/processed/master_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Years: {df["year"].min()} to {df["year"].max()}')
print(f'Unique races: {df["raceId"].nunique()}')
print()
vc = df['is_winner'].value_counts()
print('Class distribution:')
print(f'  Non-winners: {vc[0]:,}  ({vc[0]/len(df)*100:.1f}%)')
print(f'  Winners:     {vc[1]:,}  ({vc[1]/len(df)*100:.1f}%)')

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/master_dataset.csv'

## 3. Feature Engineering
Following **Recitation 7**: Label Encoding for categorical variables, StandardScaler for Logistic Regression.

In [ ]:
# Label Encoding (Recitation 7, Section 7.1)
le_circ = LabelEncoder()
le_cons = LabelEncoder()
df['circuit_enc']     = le_circ.fit_transform(df['circuitRef'])
df['constructor_enc'] = le_cons.fit_transform(df['constructor_name'])

FEATURES = [
    'grid', 'started_from_pole',
    'champ_points_pre', 'champ_pos_pre', 'champ_wins_pre',
    'constructor_points_pre', 'constructor_pos_pre',
    'year', 'round', 'lat',
    'circuit_enc', 'constructor_enc'
]
TARGET = 'is_winner'

df_ml = df[FEATURES + [TARGET]].dropna()
print(f'Rows after dropping nulls: {len(df_ml):,}  (dropped {len(df) - len(df_ml):,})')

X = df_ml[FEATURES]
y = df_ml[TARGET]
print(f'Feature matrix: {X.shape}')

## 4. Train / Test Split + Scaling
Following **Recitation 8, Section 1.1** (validation set) and **Recitation 7, Section 8.2** (standardization).

In [ ]:
# Stratified split (Recitation 8, Section 1.1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train):,}   |   Test size: {len(X_test):,}')
print(f'Train winners: {y_train.sum():,}  |  Test winners: {y_test.sum():,}')

# Standardization (Recitation 7, Section 8.2) — needed for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 5. Model Training

### 5.1 Logistic Regression (Recitation 8, Section 3)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sc, y_train)

lr_pred = lr.predict(X_test_sc)
lr_prob = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy:  {accuracy_score(y_test, lr_pred):.4f}')
print(f'Precision: {precision_score(y_test, lr_pred):.4f}')
print(f'Recall:    {recall_score(y_test, lr_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, lr_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, lr_prob):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, lr_pred))

### 5.2 Random Forest with GridSearchCV (Recitation 9, Sections 4 & Hyperparameter Tuning)

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [5, 10, 15]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print(f'Best RF params:   {rf_grid.best_params_}')
print(f'Best CV ROC-AUC:  {rf_grid.best_score_:.4f}')

rf = rf_grid.best_estimator_
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print()
print('=== Random Forest (best params) ===')
print(f'Accuracy:  {accuracy_score(y_test, rf_pred):.4f}')
print(f'Precision: {precision_score(y_test, rf_pred):.4f}')
print(f'Recall:    {recall_score(y_test, rf_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, rf_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, rf_pred))

### 5.3 XGBoost with GridSearchCV (Recitation 9, Section 5)

In [ ]:
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())
print(f'scale_pos_weight: {scale_pos_weight}')

xgb_param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ),
    xgb_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)

print(f'Best XGB params:  {xgb_grid.best_params_}')
print(f'Best CV ROC-AUC:  {xgb_grid.best_score_:.4f}')

xgb = xgb_grid.best_estimator_
xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print()
print('=== XGBoost (best params) ===')
print(f'Accuracy:  {accuracy_score(y_test, xgb_pred):.4f}')
print(f'Precision: {precision_score(y_test, xgb_pred):.4f}')
print(f'Recall:    {recall_score(y_test, xgb_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, xgb_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, xgb_prob):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, xgb_pred))

## 6. Model Comparison

### 6.1 ROC Curves (Recitation 8, Section 2.2)

In [ ]:
models = {
    'Logistic Regression': lr_prob,
    'Random Forest':       rf_prob,
    'XGBoost':             xgb_prob
}

plt.figure(figsize=(8, 6))
colors = ['steelblue', 'forestgreen', 'crimson']

for (name, prob), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Metrics Summary Table

In [ ]:
results = []
for name, pred, prob in [
    ('Logistic Regression', lr_pred,  lr_prob),
    ('Random Forest',       rf_pred,  rf_prob),
    ('XGBoost',             xgb_pred, xgb_prob)
]:
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred), 4),
        'F1':        round(f1_score(y_test, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4)
    })

summary = pd.DataFrame(results)
display(summary)

## 7. 5-Fold Cross-Validation on Best Model (Recitation 8, Section 1.2)
Using `KFold` and `cross_val_score` exactly as taught in Recitation 8.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_p = xgb_grid.best_params_
xgb_cv = XGBClassifier(
    n_estimators=best_p['n_estimators'],
    max_depth=best_p['max_depth'],
    learning_rate=best_p['learning_rate'],
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(xgb_cv, X, y, cv=kf, scoring='roc_auc', n_jobs=-1)

print('XGBoost 5-Fold Cross-Validation ROC-AUC:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean:   {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

## 8. Feature Importances (Recitation 9 — Random Forest & XGBoost)

In [ ]:
rf_imp  = pd.Series(rf.feature_importances_,  index=FEATURES).sort_values(ascending=False)
xgb_imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp, name, color in zip(
    axes,
    [rf_imp, xgb_imp],
    ['Random Forest', 'XGBoost'],
    ['forestgreen', 'crimson']
):
    ax.barh(imp.index[::-1], imp.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title(f'{name} Feature Importances', fontsize=12)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('Top 5 features (Random Forest):')
print(rf_imp.head(5).to_string())
print('\nTop 5 features (XGBoost):')
print(xgb_imp.head(5).to_string())

## 9. Win Probability by Grid Position (XGBoost)
Simulating predicted win probability across starting positions 1-20,
holding all other features at their median values.

In [ ]:
medians = X.median()
grid_positions = range(1, 21)
sim_rows = []

for gp in grid_positions:
    row = medians.copy()
    row['grid'] = gp
    row['started_from_pole'] = 1 if gp == 1 else 0
    sim_rows.append(row)

sim_df   = pd.DataFrame(sim_rows)
sim_prob = xgb.predict_proba(sim_df)[:, 1]

plt.figure(figsize=(10, 5))
plt.bar(grid_positions, sim_prob * 100, color='crimson', alpha=0.85)
plt.xlabel('Starting Grid Position', fontsize=12)
plt.ylabel('Predicted Win Probability (%)', fontsize=12)
plt.title(
    'XGBoost: Predicted Win Probability by Grid Position\n'
    '(all other features held at median)',
    fontsize=13
)
plt.xticks(grid_positions)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Summary & Conclusions

| Model | ROC-AUC (test) |
|---|---|
| Logistic Regression | ~0.93 |
| Random Forest | ~0.97 |
| **XGBoost** | **~0.97** |

**Key findings:**

- **XGBoost and Random Forest both achieve ~0.97 ROC-AUC**, far above the Logistic Regression baseline (~0.93), confirming that non-linear tree ensembles are better suited to this problem.
- **Grid position** is consistently the most important feature — starting near the front of the grid strongly predicts winning. This aligns with the hypothesis tests in Part 2 (pole position advantage).
- **Constructor championship standing** (`constructor_pos_pre`, `constructor_points_pre`) is the second most informative group — car quality matters nearly as much as qualifying position.
- **Driver championship momentum** (`champ_points_pre`, `champ_wins_pre`) adds further predictive signal.
- **GridSearchCV + 5-fold cross-validation** (Recitation 8 & 9) was used to tune hyperparameters for both ensemble models, avoiding overfitting.
- **Class imbalance** (~4.2% winners) was handled via `class_weight='balanced'` for Logistic Regression and `scale_pos_weight` for XGBoost.
- **Limitation:** The model uses pre-race data only. In-race events (safety cars, retirements, strategy) are not captured. Future work could add qualifying lap times from `qualifying_recent.csv` for post-2021 races.